
##### DEPRECATED
##### Replaced by: Silver_SQL

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import format_number
from pyspark.sql.functions import translate,lower
from pyspark.sql.functions import col

# Iniciar a SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Transformação Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [0]:
dbutils.fs.mkdirs("/Volumes/vendas_pecas/default/my/prata")

In [0]:
path = "/Volumes/vendas_pecas/default/my/"
bronze_db = "vendas_pecas.camada_bronze"
silver_db = "vendas_pecas.camada_prata"
silver = "silver/"
bronze = "prata/"

In [0]:
df_silver = spark.read.table(bronze_db + ".tb_bronze")

In [0]:
acentos     = "áàãâÁÀÃÂéèêÉÈÊíìîÍÌÎóòõôÓÒÕÔúùûÚÙÛçÇ"
sem_acentos = "aaaaAAAAeeeEEEiiiIIIooooOOOOuuuUUUcC"
#removendo acentos da tabela produto_peca
df_silver = df_silver.withColumn("produto_peca_sem_acento", translate(df_silver.produto_peca, acentos, sem_acentos))\
                    .withColumn("quantidade", col("quantidade").cast("bigint"))\
                    .withColumn("loja_nome_sem_acento", translate(df_silver.loja_nome, acentos, sem_acentos))\
                    .withColumn("grupo_loja_sem_acento", translate(df_silver.grupo_loja, acentos, sem_acentos))\
                    .withColumn("ano_venda", year(col("data_venda")))\
                    .withColumn("mes_venda", month(col("data_venda")))\
                    .withColumn("data_venda", col("data_venda").cast("date"))

In [0]:
df_silver = df_silver.dropDuplicates(["IdNotaFiscal"])


In [0]:
regra_validos = (
    col("IdNotaFiscal").isNotNull() &
    col("data_venda").isNotNull() &
    col("cliente_id").isNotNull() &
    col("cliente_nome").isNotNull() &
    col("marca_carro").isNotNull() &
    col("modelo_carro").isNotNull() &
    col("produto_peca").isNotNull()  &
    col("valor_unitario").isNotNull() & (col("valor_unitario") > 0) &
    col("quantidade").isNotNull() & (col("quantidade") > 0) &
    col("custo_unitario").isNotNull() & (col("custo_unitario") > 0) &
    col("loja_id").isNotNull() &
    col("loja_nome").isNotNull() &
    col("grupo_loja").isNotNull()&
    col("vendedor_id").isNotNull() &
    col("vendedor_nome").isNotNull()
)

In [0]:
df_validos = df_silver.filter(regra_validos)
df_invalidos = df_silver.filter(~regra_validos)

In [0]:
df_invalidos.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("vendas_pecas.camada_prata.tb_quarentena")

In [0]:
df_validos.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("vendas_pecas.camada_prata.tb_prata")